<a href="https://colab.research.google.com/github/Koshalendar/BIS_LAB-/blob/main/Particle_Swarm_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

def pso_scheduler(num_staff, num_days, num_shifts, req_matrix, S, max_iter, w=0.5, c1=1.5, c2=1.5):
    D = num_staff * num_days

    def evaluate_conflicts(particle_flat):
        grid = [particle_flat[i * num_days:(i + 1) * num_days] for i in range(num_staff)]
        conflicts = 0
        for day in range(num_days):
            shift_counts = [0] * num_shifts
            for staff in range(num_staff):
                shift = int(grid[staff][day])
                if shift < num_shifts:
                    shift_counts[shift] += 1
            for shift in range(num_shifts):
                conflicts += abs(shift_counts[shift] - req_matrix[day][shift])
        return conflicts

    x = [[float(random.randint(0, num_shifts - 1)) for _ in range(D)] for _ in range(S)]
    v = [[random.uniform(-1, 1) for _ in range(D)] for _ in range(S)]
    p = [row[:] for row in x]

    g = min(p, key=evaluate_conflicts)
    g_fitness = evaluate_conflicts(g)

    for _ in range(max_iter):
        for i in range(S):
            for d in range(D):
                r1, r2 = random.random(), random.random()
                v[i][d] = w * v[i][d] + c1 * r1 * (p[i][d] - x[i][d]) + c2 * r2 * (g[d] - x[i][d])
                val = round(x[i][d] + v[i][d])
                x[i][d] = float(max(0, min(num_shifts - 1, val)))

            if evaluate_conflicts(x[i]) < evaluate_conflicts(p[i]):
                p[i] = x[i][:]
                if evaluate_conflicts(p[i]) < g_fitness:
                    g = p[i][:]
                    g_fitness = evaluate_conflicts(g)

    return g, g_fitness

if __name__ == "__main__":
    num_staff = int(input("Enter number of staff (docs + nurses): "))
    num_days = int(input("Enter number of schedule days: "))
    num_shifts = int(input("Enter number of shifts types (e.g., 3 for Morning, Evening, Night): "))

    req_matrix = []
    print("Enter staffing requirements for each shift day by day:")
    for day in range(num_days):
        day_reqs = []
        for shift in range(num_shifts):
            val = int(input(f"  Day {day+1}, Shift {shift} requirement: "))
            day_reqs.append(val)
        req_matrix.append(day_reqs)

    S = int(input("Enter PSO Swarm size (e.g., 30): "))
    max_iter = int(input("Enter maximum PSO iterations: "))

    best_flat, final_conflicts = pso_scheduler(num_staff, num_days, num_shifts, req_matrix, S, max_iter)

    print("\n--- Optimized Schedule Matrix (Rows=Staff, Cols=Days, Vals=Shift ID) ---")
    for idx in range(num_staff):
        print(f"Staff member {idx+1}: {[int(val) for val in best_flat[idx*num_days : (idx+1)*num_days]]}")
    print("Remaining staff allocation mismatches:", final_conflicts)


Enter number of staff (docs + nurses): 3
Enter number of schedule days: 2
Enter number of shifts types (e.g., 3 for Morning, Evening, Night): 2
Enter staffing requirements for each shift day by day:
  Day 1, Shift 0 requirement: 2
  Day 1, Shift 1 requirement: 1
  Day 2, Shift 0 requirement: 1
  Day 2, Shift 1 requirement: 2
Enter PSO Swarm size (e.g., 30): 30
Enter maximum PSO iterations: 100

--- Optimized Schedule Matrix (Rows=Staff, Cols=Days, Vals=Shift ID) ---
Staff member 1: [1, 1]
Staff member 2: [0, 0]
Staff member 3: [0, 1]
Remaining staff allocation mismatches: 0
